# Customizing the scripts: run your own reaction

Notebooks `01`–`04` compare the numerical, QSSA, and Gillespie methods on fixed
mechanisms. This notebook shows how to express
**any** mechanism as the block the standalone scripts read, so you can run your
own reaction.

The repository root holds four self-contained scripts:

| script | what it computes |
| --- | --- |
| `numerical.py` | steady state by stiff ODE integration (the reference) |
| `qssa.py` | steady state by the algebraic quasi-steady-state approximation |
| `gillespie.py` | stochastic (Gillespie Monte Carlo) current, with error bars |
| `transient.py` | potentiostatic current-time response |

Each has a `MECHANISM (edit me)` block near the top. Edit that block for your
reaction and run the file, for example:

```
python numerical.py
```

All four scripts read the **same** block. The sections below give the ready-to-paste block
for each of the four reactions in the paper.

## The MECHANISM block, field by field

```python
SPECIES = ["*", "H"]      # every surface species, including the free site
SITE    = "*"             # the free site; its coverage is fixed by sum(theta) = 1
DIRECTION = -1            # -1 = reduction, scanned at eta < 0;  +1 = oxidation, eta > 0

STEPS = [
    dict(reactants={"*": 1}, products={"H": 1},
         n_e=1, beta=0.5, k0=1e-3, km0=1.0, label="Volmer"),
    # ... one dict per elementary step ...
]

ETAS = np.linspace(-0.05, -0.40, 8)   # overpotentials to scan (V)
```

Per step:

- **`reactants` / `products`** — `{species: coefficient}`. A coefficient of `2`
  is a bimolecular term (it enters the rate as $\theta^2$).
- **`n_e`** — electrons the electrode transfers in this step; `0` for a chemical
  (non-electrochemical) step.
- **`beta`** — symmetry factor of an electrochemical step (usually `0.5`); use
  `0.0` for a chemical step.
- **`k0` / `km0`** — standard forward / reverse rate constants.
- **`a_fwd` / `a_rev`** *(optional, default `1.0`)* — a constant bulk-activity
  factor on the forward / reverse rate, e.g. an H⁺ or OH⁻ activity to set a pH.
- **`label`** *(optional)* — a name for the step, used in the printout.

Two conventions worth knowing:

- The steps are written in the **forward** direction set by `DIRECTION`. the
  Butler–Volmer factors automatically favor the reverse rates on the other
  branch, so the *same* steps describe both a reduction and its reverse
  oxidation. That is exactly how HER and HOR share one block.
- If the product of the step equilibrium constants $K_i = k_{0,i}/k_{m0,i}$ over
  the cycle equals 1, the current is zero at $\eta = 0$ (thermodynamic
  consistency). The unit placeholder constants in the ORR and OER blocks below
  satisfy this. Replace them with your own energetics.

## 1. HER — acid hydrogen evolution (cathodic)

$$\text{Volmer:}\quad *+\mathrm{H^+}+e^-\rightleftharpoons \mathrm{H^*}$$
$$\text{Heyrovsky:}\quad \mathrm{H^*}+\mathrm{H^+}+e^-\rightleftharpoons *+\mathrm{H_2}$$
$$\text{Tafel:}\quad 2\,\mathrm{H^*}\rightleftharpoons 2\,*+\mathrm{H_2}\;\;\text{(chemical)}$$

```python
SPECIES = ["*", "H"]
SITE    = "*"
DIRECTION = -1

STEPS = [
    dict(reactants={"*": 1}, products={"H": 1}, n_e=1, beta=0.5, k0=1e-3, km0=1.0,  label="Volmer"),
    dict(reactants={"H": 1}, products={"*": 1}, n_e=1, beta=0.5, k0=1.0,  km0=1e-3, label="Heyrovsky"),
    dict(reactants={"H": 2}, products={"*": 2}, n_e=0, beta=0.0, k0=1.0,  km0=1e-6, label="Tafel"),
]

ETAS = np.linspace(-0.05, -0.40, 8)
```

This is the default already in every script, so `python numerical.py` runs HER
out of the box.

## 2. HOR — hydrogen oxidation (anodic)

HOR is the **same mechanism as HER**, run on the anodic branch. Keep the HER
block above and change only two lines to flip the direction and scan positive
overpotentials:

```python
DIRECTION = +1
ETAS = np.linspace(+0.01, +0.60, 8)
```

Because the steps are written cathodic-forward, the Butler–Volmer factors favor
the reverse (anodic) rates at $\eta > 0$. No step needs rewriting.

## 3. ORR — acid oxygen reduction (cathodic, 5 steps)

$$1.\;\mathrm{M}+\mathrm{O_2}\rightleftharpoons \mathrm{MOO}\;\;\text{(ChemAds, chemical)}$$
$$2.\;\mathrm{MOO}+\mathrm{H^+}+e^-\rightleftharpoons \mathrm{MOOH}\;\;\text{(PCET1)}$$
$$3.\;\mathrm{MOOH}+\mathrm{H^+}+e^-\rightleftharpoons \mathrm{MO}\;\;\text{(PCET2)}$$
$$4.\;\mathrm{MO}+\mathrm{H^+}+e^-\rightleftharpoons \mathrm{MOH}\;\;\text{(PCET3)}$$
$$5.\;\mathrm{MOH}+\mathrm{H^+}+e^-\rightleftharpoons \mathrm{M}\;\;\text{(PCET4)}$$

The free site is the bare metal `M`. Step 1 is chemical (`n_e=0`). The four
proton-coupled electron transfers each carry one electron.

```python
SPECIES = ["M", "MOO", "MOOH", "MO", "MOH"]
SITE    = "M"
DIRECTION = -1

STEPS = [
    dict(reactants={"M": 1},    products={"MOO": 1},  n_e=0, beta=0.0, k0=1.0, km0=1.0, label="ChemAds"),
    dict(reactants={"MOO": 1},  products={"MOOH": 1}, n_e=1, beta=0.5, k0=1.0, km0=1.0, label="PCET1"),
    dict(reactants={"MOOH": 1}, products={"MO": 1},   n_e=1, beta=0.5, k0=1.0, km0=1.0, label="PCET2"),
    dict(reactants={"MO": 1},   products={"MOH": 1},  n_e=1, beta=0.5, k0=1.0, km0=1.0, label="PCET3"),
    dict(reactants={"MOH": 1},  products={"M": 1},    n_e=1, beta=0.5, k0=1.0, km0=1.0, label="PCET4"),
]

ETAS = np.linspace(-0.02, -0.55, 12)
```

The unit `k0`/`km0` above are balanced placeholders. Set each step's constants
from your own intermediate energies. The paper's designed rate-limiting-step
cases (`mechanisms.py`) are built from this same block by adjusting only the
constants.

## 4. OER — alkaline oxygen evolution (anodic, 5 steps)

$$1.\;\mathrm{M}+\mathrm{OH^-}\rightleftharpoons \mathrm{MOH}+e^-\;\;\text{(ET1)}$$
$$2.\;\mathrm{MOH}+\mathrm{OH^-}\rightleftharpoons \mathrm{MO}+\mathrm{H_2O}+e^-\;\;\text{(ET2)}$$
$$3.\;\mathrm{MO}+\mathrm{OH^-}\rightleftharpoons \mathrm{MOOH}+e^-\;\;\text{(ET3)}$$
$$4.\;\mathrm{MOOH}+\mathrm{OH^-}\rightleftharpoons \mathrm{MOO^-}+\mathrm{H_2O}\;\;\text{(ChemPT, chemical)}$$
$$5.\;\mathrm{MOO^-}\rightleftharpoons \mathrm{M}+\mathrm{O_2}+e^-\;\;\text{(ET4)}$$

Anodic (`DIRECTION=+1`), scanned at $\eta>0$. Step 4 (ChemPT) is chemical
(`n_e=0`). `MOOm` is the name used for the $\mathrm{MOO^-}$ intermediate.

```python
SPECIES = ["M", "MOH", "MO", "MOOH", "MOOm"]
SITE    = "M"
DIRECTION = +1

STEPS = [
    dict(reactants={"M": 1},    products={"MOH": 1},  n_e=1, beta=0.5, k0=1.0, km0=1.0, label="ET1"),
    dict(reactants={"MOH": 1},  products={"MO": 1},   n_e=1, beta=0.5, k0=1.0, km0=1.0, label="ET2"),
    dict(reactants={"MO": 1},   products={"MOOH": 1}, n_e=1, beta=0.5, k0=1.0, km0=1.0, label="ET3"),
    dict(reactants={"MOOH": 1}, products={"MOOm": 1}, n_e=0, beta=0.0, k0=1.0, km0=1.0, label="ChemPT"),
    dict(reactants={"MOOm": 1}, products={"M": 1},    n_e=1, beta=0.5, k0=1.0, km0=1.0, label="ET4"),
]

ETAS = np.linspace(+0.02, +0.55, 12)
```

To make the rates depend on pH, put the OH⁻ activity on the electrochemical
steps (e.g. `a_fwd=a_OH` on ET1–ET3 (and the ChemPT forward), with `a_OH` your
hydroxide activity).

## Running the scripts

With a block in place, run any of the four from the repository root:

```
python numerical.py     # steady-state table (j, coverages, local Tafel slope)
python qssa.py          # same, via the algebraic QSSA (fast)
python gillespie.py     # stochastic j with a standard error
python transient.py     # current-time response after a potential step
```

Each prints a table like the ones in notebooks `01`–`04`. Two switches at the
top of each script control the extras:

- `CSV_OUT = "numerical.csv"` — filename to write, or `None` to skip.
- `SHOW_PLOT = True` — pop a $\log|j|$-vs-$\eta$ window (needs matplotlib).
  Set `False` for headless runs.

That is the whole workflow: one mechanism block, four methods!